# Counterfactual on a star multi-table dataset

In [1]:
# Automatically reload external modules when they are modified
%load_ext autoreload
%autoreload 2

In [2]:
# Import relevant packages
import os
import pandas as pd
import warnings

from khiops import core as kh

import sys
sys.path.append('./modules')
from counterfactual import find_counterfactual_trajectory_using_core_api
from counterfactual_helper import build_aggregates_dataframe, build_monotable_dictionary_domain

In [3]:
# Show all columns of a dataframe
pd.set_option('display.max_columns', None)

In [4]:
# Ignore warnings
warnings.filterwarnings("ignore", category=UserWarning)

## Load datasets

In [5]:
sepsis_dataset_dir = ("./")
sepsis_file_path = os.path.join(sepsis_dataset_dir, "S_PatientsWithLabels-sep-tab.txt")
sepsis_df = pd.read_csv(sepsis_file_path, sep="\t", encoding="utf-8")
logs_file_path = os.path.join(sepsis_dataset_dir, "S_PatientsWithFeatures_4logs_sep-tab.txt")
logs_df = pd.read_csv(logs_file_path, sep="\t", encoding="utf-8", keep_default_na=False) # None must be kept

In [6]:
sepsis_df.head()

,Patient_ID,Target
0,100008.0,0
1,10001.0,0
2,100013.0,1
3,100016.0,1
4,100047.0,1


In [7]:
logs_df.head()

,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,FiO2,pH,PaCO2,SaO2,AST,BUN,Alkalinephos,Calcium,Chloride,Creatinine,Bilirubin_direct,Glucose,Lactate,Magnesium,Phosphate,Potassium,TroponinI,Hct,PTT,WBC,Fibrinogen,Platelets,Age,HospAdmTime,Patient_ID
0,87.1,97.7,36.10,118.1,71.8,56.8,19.5,35.9,-15.06,19.59,0.4,7.398,35.0,97.18,51.2,28.4,67.6,2.518,104.0,1.692,0.364,114.4,1.716,2.00,2.82,5.44,120.238,29.94,35.02,13.10,223.4,173.2,65.0,-211.64,100008.0
1,80.0,95.5,36.20,125.0,74.0,57.0,16.0,34.5,-7.24,19.59,0.4,7.398,35.0,97.18,146.2,25.6,72.0,2.518,104.4,1.438,0.318,128.0,1.716,2.10,2.82,5.44,80.420,29.94,34.84,12.40,223.4,193.6,65.0,-211.64,100008.0
2,85.0,99.0,36.10,137.0,79.0,61.0,16.0,36.0,-7.24,19.59,0.4,7.398,35.0,96.38,146.2,25.6,72.0,2.518,104.4,1.438,0.318,122.0,1.716,2.10,2.82,5.44,80.420,29.94,34.84,12.40,223.4,193.6,65.0,-211.64,100008.0
3,71.0,99.6,35.80,175.0,83.0,59.0,18.0,31.3,-11.74,19.59,0.4,7.398,35.0,96.38,146.2,25.8,72.0,2.518,104.4,1.282,0.364,119.4,1.716,2.12,2.82,5.44,80.420,30.56,36.00,12.02,223.4,147.8,65.0,-211.64,100008.0
4,107.5,95.0,37.17,108.5,86.5,74.5,16.0,21.8,0.00,22.00,0.5,7.460,33.0,96.80,25.2,8.0,65.4,8.200,113.0,1.200,5.420,130.0,1.400,1.50,2.34,3.30,7.840,30.30,33.02,11.00,205.2,172.6,87.2,-102.37,10001.0


## Train a Khiops classifier

The Khiops core API is used

In [8]:
dictionary_file_path = os.path.join(sepsis_dataset_dir, "Sepsis.kdic")
report_file_path = os.path.join("kh_samples", "train_predictor_file_path", "AnalysisResults.khj") , model_dictionary_file_path = kh.train_predictor(
    dictionary_file_path,
    "PatientWithLabels",
    sepsis_file_path,
    "Target",
    report_file_path,
    additional_data_tables={"LogPatient": logs_file_path},
    max_trees=0,
    max_constructed_variables=100 # to avoid variable block
)
print("Report file available at " + report_file_path)
print("Model dictionary file available at " + model_dictionary_file_path)

KhiopsRuntimeError: khiops execution had errors (return code 1):
Errors in log:
Line 1: error : File ./Sepsis.kdic : Unable to open file (No such file or directory)
Line 2: error : Database : Dictionary PatientWithLabels not found
Line 3: error : Input command file c:\Users\voyl6404\AppData\Local\Temp\1\khiops\python\train_predictor_vicfa_at._kh, line 12 : Invalid command (TrainDatabase.DatabaseSpec.Data.DatabaseFiles.List.Key LogPatient); in 'Database files', the 'Data path' value 'LogPatient' is invalid and should be one of ('')
Line 4: error : Input command file c:\Users\voyl6404\AppData\Local\Temp\1\khiops\python\train_predictor_vicfa_at._kh : Analysis of input commands interrupted because of errors

Fatal errors in log:
Line 5: fatal error : Command file : Batch mode failure

Contents of stderr:
fatal error : Command file : Batch mode failure



## Get the aggregates used the Khiops multi-table classifier

In [28]:
# Build aggregates
na_values = ["", "#N/A", "#N/A N/A", "#NA", "-1.#IND", "-1.#QNAN", "-NaN", "-nan", "1.#IND", "1.#QNAN", "<NA>", "N/A", "NA", "NULL", "NaN", "n/a", "nan", "null"] # 'None' is removed
accident_aggregates_df = build_aggregates_dataframe(sepsis_df, logs_df, model_dictionary_file_path, 'SNB_PatientWithLabels', 'LogPatient', keep_default_na=False, na_values=na_values)
accident_aggregates_df.head()

,Patient_ID,Max(LogPatient.BUN),Max(LogPatient.HospAdmTime),Max(LogPatient.Lactate),Max(LogPatient.O2Sat),Max(LogPatient.PaCO2),Max(LogPatient.Phosphate),Max(LogPatient.SaO2),Max(LogPatient.Temp),Max(LogPatient.WBC),Mean(LogPatient.Chloride),Mean(LogPatient.EtCO2),Mean(LogPatient.HCO3),Mean(LogPatient.HR),Mean(LogPatient.Platelets),Mean(LogPatient.pH),Median(LogPatient.BUN),Median(LogPatient.Calcium),Median(LogPatient.FiO2),Median(LogPatient.HCO3),Median(LogPatient.HospAdmTime),Median(LogPatient.O2Sat),Median(LogPatient.Resp),Median(LogPatient.SBP),Median(LogPatient.Temp),Min(LogPatient.BaseExcess),Min(LogPatient.Bilirubin_direct),Min(LogPatient.Potassium),Min(LogPatient.SaO2),Min(LogPatient.Temp),Min(LogPatient.WBC),Min(LogPatient.pH),StdDev(LogPatient.Alkalinephos),StdDev(LogPatient.Creatinine),StdDev(LogPatient.EtCO2),StdDev(LogPatient.HR),StdDev(LogPatient.Hct),StdDev(LogPatient.Lactate),StdDev(LogPatient.Magnesium),StdDev(LogPatient.O2Sat),StdDev(LogPatient.PTT),StdDev(LogPatient.Platelets),StdDev(LogPatient.Potassium),Sum(LogPatient.BUN),Sum(LogPatient.Creatinine),Sum(LogPatient.HospAdmTime),Sum(LogPatient.Magnesium),Sum(LogPatient.PaCO2),Sum(LogPatient.Platelets)
0,100008.0,28.4,-211.64,1.716,99.6,35.0,2.82,97.18,36.20,13.10,104.30,34.425,19.5900,80.775,177.05,7.3980,25.7,2.518,0.40,19.59,-211.64,98.35,17.00,131.00,36.100,-15.06,0.318,5.44,96.38,35.800,12.02,7.398,1.905256,0.147013,1.899177,6.204988,0.268468,0.000000,0.046904,1.572419,0.481949,18.829432,0.000000,105.4,5.850,-846.56,8.32,140.0,708.2
1,10001.0,12.0,-102.37,3.180,100.0,33.8,2.94,96.80,37.17,13.60,111.35,31.775,21.6000,94.125,172.15,7.4655,10.7,8.200,0.50,21.60,-102.37,98.00,19.50,118.25,36.884,0.00,5.280,3.30,96.60,36.854,11.00,7.460,18.937265,0.000000,5.795418,11.081375,0.784984,0.842971,0.242487,1.920286,0.000000,3.667083,0.522948,41.4,4.800,-409.48,7.68,127.4,688.6
2,100013.0,24.4,-253.56,1.716,98.0,35.0,3.00,96.90,38.00,12.12,104.65,26.875,19.6125,118.150,209.80,7.3980,24.4,2.518,0.40,19.59,-253.56,98.00,20.40,120.30,37.870,-15.06,0.364,4.96,96.38,37.600,12.12,7.398,0.000000,0.000000,2.965953,9.250270,0.000000,0.000000,0.000000,0.173205,0.000000,0.000000,0.058949,97.6,4.904,-1014.24,8.72,140.0,839.2
3,100016.0,32.0,-7.52,2.854,100.0,38.8,4.18,97.18,37.38,14.06,103.00,27.400,19.5425,106.750,171.20,7.3905,21.6,6.970,0.54,19.59,-7.52,100.00,16.25,129.70,37.140,-15.06,0.318,4.20,97.18,36.520,9.90,7.338,4.600000,2.340867,0.504975,2.861381,1.056160,0.099311,0.213776,0.519615,0.000000,42.262040,0.210891,96.8,15.342,-30.08,11.08,146.8,684.8
4,100047.0,29.0,-151.79,1.716,100.0,35.0,2.82,97.18,38.60,13.10,104.40,35.550,19.5900,87.500,183.40,7.3980,27.3,3.972,0.40,19.59,-151.79,98.75,12.50,91.50,38.400,-11.74,0.318,5.14,97.18,38.000,12.40,7.398,1.905256,0.071000,1.386542,2.291288,0.307042,0.000000,0.010000,1.556237,1.567506,10.200000,0.000000,109.2,6.036,-607.16,8.44,140.0,733.6


## Build a mono-table dictionary

In [29]:
monotable_dictionary_domain = build_monotable_dictionary_domain(model_dictionary_file_path, 'SNB_PatientWithLabels')

## Counterfactual

### Build individual

In [30]:
def build_individual(Patient_ID: int):
    individual = accident_aggregates_df[accident_aggregates_df['Patient_ID'] == Patient_ID]
    return individual

In [34]:
# Build individual (using the 'Patient_Id')
individual = build_individual(10001.0)
individual

,Patient_ID,Max(LogPatient.BUN),Max(LogPatient.HospAdmTime),Max(LogPatient.Lactate),Max(LogPatient.O2Sat),Max(LogPatient.PaCO2),Max(LogPatient.Phosphate),Max(LogPatient.SaO2),Max(LogPatient.Temp),Max(LogPatient.WBC),Mean(LogPatient.Chloride),Mean(LogPatient.EtCO2),Mean(LogPatient.HCO3),Mean(LogPatient.HR),Mean(LogPatient.Platelets),Mean(LogPatient.pH),Median(LogPatient.BUN),Median(LogPatient.Calcium),Median(LogPatient.FiO2),Median(LogPatient.HCO3),Median(LogPatient.HospAdmTime),Median(LogPatient.O2Sat),Median(LogPatient.Resp),Median(LogPatient.SBP),Median(LogPatient.Temp),Min(LogPatient.BaseExcess),Min(LogPatient.Bilirubin_direct),Min(LogPatient.Potassium),Min(LogPatient.SaO2),Min(LogPatient.Temp),Min(LogPatient.WBC),Min(LogPatient.pH),StdDev(LogPatient.Alkalinephos),StdDev(LogPatient.Creatinine),StdDev(LogPatient.EtCO2),StdDev(LogPatient.HR),StdDev(LogPatient.Hct),StdDev(LogPatient.Lactate),StdDev(LogPatient.Magnesium),StdDev(LogPatient.O2Sat),StdDev(LogPatient.PTT),StdDev(LogPatient.Platelets),StdDev(LogPatient.Potassium),Sum(LogPatient.BUN),Sum(LogPatient.Creatinine),Sum(LogPatient.HospAdmTime),Sum(LogPatient.Magnesium),Sum(LogPatient.PaCO2),Sum(LogPatient.Platelets)
1,10001.0,12.0,-102.37,3.18,100.0,33.8,2.94,96.8,37.17,13.6,111.35,31.775,21.6,94.125,172.15,7.4655,10.7,8.2,0.5,21.6,-102.37,98.0,19.5,118.25,36.884,0.0,5.28,3.3,96.6,36.854,11.0,7.46,18.937265,0.0,5.795418,11.081375,0.784984,0.842971,0.242487,1.920286,0.0,3.667083,0.522948,41.4,4.8,-409.48,7.68,127.4,688.6


### Find counterfactual trajectory

In [35]:
# Find counterfactual trajectory
alterable_variables = individual.columns.to_list()
trajectories = find_counterfactual_trajectory_using_core_api(monotable_dictionary_domain, report_file_path, 'SNB_PatientWithLabels', individual, alterable_variables)
print(trajectories[0])
trajectories[0].table

Initial class = '0' - Proba '0' = 0.561962 - Proba indiv. = 0.000000
A counterfactual has been found
Step 1 : 'StdDev(LogPatient.EtCO2)' from '5.795418449' to '0.2537' ]-∞, 0.5074] --> Proba '0' = 0.131336, Proba indiv. = 0.000000
